# 02 Agent Sandbox -- 逐个工具/Agent 单独调试

本 notebook 用于逐个测试子 Agent 的结构化输出、工具函数、完整 Agent 对话流程。

| Cell | 内容 |
|---|---|
| 1 | 初始化 LLM |
| 2 | 测试出题 Agent (QuizOutput) |
| 3 | 测试评估 Agent (EvaluationOutput) |
| 4 | 测试错题整理 Agent (ConsolidationOutput) |
| 5 | 测试 review 变式题 |
| 6 | 测试 search_knowledge_base 工具 |
| 7 | quiz_flow 完整流程说明 |
| 8 | 测试 create_agent 完整流程 (含 interrupt/resume) |

In [ ]:
# Cell 1: 初始化 LLM
from teaching_agent.agents.llm import get_llm, structured_call
from teaching_agent.agents.schemas import QuizOutput, EvaluationOutput, ConsolidationOutput
from teaching_agent.agents.prompts import QUIZ_PROMPT, EVALUATION_PROMPT, CONSOLIDATION_PROMPT

llm = get_llm()
print("✓ LLM 初始化成功")

In [ ]:
# Cell 2: 测试出题 Agent
prompt = QUIZ_PROMPT.format(
    course_name="计算机网络",
    topic="TCP三次握手",
    conversation_context="学生刚学习了TCP连接建立过程",
    retrieved_chunks="客户端发送SYN，服务端回复SYN-ACK...",
    weak_points="无",
    available_kps="TCP三次握手, TCP四次挥手, UDP协议",
    review_context="",
    variant_requirement="",
)
quiz = await structured_call(llm, QuizOutput, prompt)
print(f"题型: {quiz.q_type}")
print(f"题干: {quiz.stem}")
print(f"选项: {quiz.options}")
print(f"答案: {quiz.answer}")
print(f"知识点: {quiz.kp_name}")

In [ ]:
# Cell 3: 测试评估 Agent
prompt = EVALUATION_PROMPT.format(
    question_stem=quiz.stem,
    q_type=quiz.q_type,
    correct_answer=quiz.answer,
    student_answer=quiz.options[0] if quiz.options else "不知道",  # 故意选第一个
    kp_name=quiz.kp_name,
    retrieved_chunks="三次握手第二步是SYN-ACK",
)
result = await structured_call(llm, EvaluationOutput, prompt)
print(f"判定: {result.verdict}")
print(f"错因: {result.diagnosis}")
print(f"正确答案: {result.correct_answer}")
print(f"后续方向: {result.follow_up_hint}")

In [ ]:
# Cell 4: 测试错题整理 Agent
prompt = CONSOLIDATION_PROMPT.format(
    full_conversation="学生问了TCP三次握手，答错了一道题",
    answer_records="题目: 三次握手第二步是什么? 学生答: SYN 判定: wrong",
    old_profile_summary="新学生",
    old_weak_points="无",
)
consolidation = await structured_call(llm, ConsolidationOutput, prompt)
print(f"标题: {consolidation.title}")
print(f"摘要: {consolidation.summary}")
print(f"错题: {len(consolidation.mistakes)} 个")
for m in consolidation.mistakes:
    print(f"  - {m.kp_name}: {m.title}")
print(f"画像: {consolidation.profile_summary}")
print(f"薄弱点: {consolidation.weak_points}")

In [ ]:
# Cell 5: 测试 review 变式题
prompt = QUIZ_PROMPT.format(
    course_name="计算机网络",
    topic="TCP三次握手",
    conversation_context="学生正在重练错题",
    retrieved_chunks="三次握手过程...",
    weak_points="TCP三次握手",
    available_kps="TCP三次握手, TCP四次挥手, UDP协议",
    review_context="原题：三次握手第二步是什么？\n出变式题：同知识点，不同考法、不同角度。",
    variant_requirement="",
)
variant_quiz = await structured_call(llm, QuizOutput, prompt)
print(f"变式题: {variant_quiz.stem}")
print(f"题型: {variant_quiz.q_type}")
print(f"（与原题考法不同）")

In [ ]:
# Cell 6: 测试 search_knowledge_base 工具
from teaching_agent.agents.tools import make_search_tool

search = make_search_tool("free", {})
result = await search.ainvoke({"query": "TCP三次握手"})
print(result[:500])

In [ ]:
# Cell 7: quiz_flow 完整流程说明
# quiz_flow 的完整测试需要 create_agent + checkpointer（interrupt 需要 checkpointer 支持）
# 见 Cell 8 的完整 Agent 测试
print("quiz_flow 的完整测试需要 create_agent + checkpointer，见 Cell 8")

In [ ]:
# Cell 8: 完整 Agent 测试（含 interrupt/resume）
import uuid
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from langgraph.types import Command
from teaching_agent.agents.free_agent import create_free_agent
from teaching_agent.db import async_session_factory
from teaching_agent.models import Session as SessionModel
from teaching_agent.config import settings
from datetime import datetime, timezone

# 创建会话记录
session_id = f"sandbox-{uuid.uuid4().hex[:8]}"
async with async_session_factory() as s:
    s.add(SessionModel(
        id=session_id, user_id="demo-student", type="free",
        status="active", started_at=datetime.now(timezone.utc),
    ))
    await s.commit()
print(f"✓ 会话创建: {session_id}")

# 创建 checkpointer + Agent
# AsyncPostgresSaver 用 psycopg（非 asyncpg），URL 去掉 +asyncpg
db_url = settings.DATABASE_URL.replace("+asyncpg", "")
async with AsyncPostgresSaver.from_conn_string(db_url) as checkpointer:
    await checkpointer.setup()

    agent = await create_free_agent(checkpointer, session_id)
    config = {
        "configurable": {"thread_id": session_id},
        "recursion_limit": settings.RECURSION_LIMIT,
    }

    # 第一轮对话：让学生试试
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "考考我TCP三次握手"}]},
        config=config,
    )

    # 检查是否被 interrupt（出题了）
    state = await agent.aget_state(config)
    if state.next:
        # 被 interrupt 了，获取题目
        quiz_data = state.tasks[0].interrupt.value
        print(f"\n📝 题目: {quiz_data['stem']}")
        if quiz_data.get('options'):
            for i, opt in enumerate(quiz_data['options']):
                print(f"  {chr(65+i)}. {opt}")

        # 提交答案（选第二个选项）
        answer = quiz_data['options'][1] if quiz_data.get('options') else "不知道"
        print(f"\n学生作答: {answer}")
        result = await agent.ainvoke(
            Command(resume={"answer": answer}),
            config=config,
        )
        reply = result["messages"][-1].content
        print(f"\n🤖 Agent回复: {reply}")
    else:
        # 未出题，直接回复
        reply = result["messages"][-1].content
        print(f"\n🤖 Agent回复: {reply}")